# Session 1b: fMRI Quality Control and Preprocessing

In [sess-1a](sess-1a.ipynb) we learned how raw scanner data are converted to NIfTI and organised into a **BIDS** dataset. We also saw that a BIDS-compliant dataset unlocks a whole ecosystem of **BIDS Apps**: container-packaged pipelines that take a BIDS folder as input and produce standardised derivatives as output.

This session puts two of those BIDS Apps to work:

  1. **MRIQC**, to assess the **quality** of the raw data *before* any processing.

  2. **fMRIprep**, to **preprocess** the fMRI data, and to re-assess quality *after* preprocessing using its HTML report.

The lesson is organised around **two main themes**:

  1. **Quality Control** of raw BIDS data with MRIQC

  2. **Preprocessing** of fMRI data with fMRIprep

> 💡 **About running MRIQC and fMRIprep.** Both tools are *heavy*: fMRIprep typically runs for several hours per subject and uses a lot of RAM and disk. For that reason, **we will not run either in class**. Instead we look at pre-computed outputs that the OpenNeuro community publishes alongside the raw data. At the end of the notebook you will run fMRIprep yourself at home on a single subject, as an exercise.

Icons used throughout, identical to sess-1a:

| Icon | Meaning |
|---|---|
| 🖥️ | **Switch to Neurodesktop**: you will need to open the Neurodesktop environment for this section |
| 🐍 | **Python**: this section contains code cells to run in the notebook |
| 📝 | **Practice**: exercises and tasks for you to complete |


## 1. The dataset

For this session we use **[ds000117](https://openneuro.org/datasets/ds000117)**, a BIDS dataset by Wakeman & Henson (2015). Sixteen adults viewed images of **famous faces**, **unfamiliar faces**, and **scrambled faces** while being scanned: an event-related fMRI design with 9 runs per subject. It is a canonical dataset for teaching first-level GLM analysis, which we will come back to in the next session.

For our purposes today, the key point is that ds000117 also has **pre-computed MRIQC and fMRIprep derivatives** published by the [OpenNeuroDerivatives](https://github.com/OpenNeuroDerivatives) community. We do not need to run either pipeline ourselves: we just inspect their outputs.

> **A BIDS dataset can hold its own derivatives.** The BIDS standard reserves a special `derivatives/` subfolder inside a BIDS dataset for the outputs of processing pipelines (`sub-XX/`, `derivatives/mriqc/`, `derivatives/fmriprep/`, ...). This is exactly what MRIQC and fMRIprep do by default: given a BIDS input folder, they write their results inside `<bids_root>/derivatives/<pipeline_name>/`, one subfolder per pipeline.
>
> We will mirror that layout in this notebook: at the end of the three copy steps below, your `results/sess-1b/ds000117/` will look like a small, complete BIDS dataset with its own `derivatives/mriqc/` and `derivatives/fmriprep/` folders, exactly where those tools would have placed them if you had run them yourself.

### 1.1 🐍 Get the raw BIDS data

The raw dataset and its derivatives are pre-staged for you on the shared course storage at `/data/teaching/costantinoai/sess-1b/ds000117/`. In this first section we copy **only the raw part of the BIDS folder** (everything *except* `derivatives/`) into your personal `results/sess-1b/` directory. We will populate `derivatives/mriqc/` and `derivatives/fmriprep/` later, in Sections 2 and 3 respectively.

In [ ]:
# `%pip` is a notebook magic that installs Python packages into the current
# kernel. We need nilearn (plotting + stats on brain images) and pybids
# (query BIDS datasets by entity), both already seen in sess-1a.
%pip install nilearn pybids

The cell below performs a recursive copy of the raw BIDS dataset from the shared course folder into your personal results folder (in `/results/sess-1b/ds000117`). 

In [ ]:
%%bash
# `%%bash` at the top of a cell tells Jupyter to run its contents as a shell
# (terminal) script instead of Python. We use this to copy data from the shared
# course folder into our personal results/ folder

# Remove any previous copy of results/sess-1b (fresh start every time this cell runs)
rm -rf results/sess-1b

# Create the destination folder
mkdir -p results/sess-1b/ds000117

# Copy only the raw part of the BIDS dataset: the per-subject folders, the
# stimuli folder, and the top-level metadata files. 
cp -r /data/teaching/costantinoai/sess-1b/ds000117/sub-03    results/sess-1b/ds000117/
cp -r /data/teaching/costantinoai/sess-1b/ds000117/sub-05    results/sess-1b/ds000117/
cp -r /data/teaching/costantinoai/sess-1b/ds000117/stimuli   results/sess-1b/ds000117/
cp    /data/teaching/costantinoai/sess-1b/ds000117/*.json    results/sess-1b/ds000117/
cp    /data/teaching/costantinoai/sess-1b/ds000117/*.tsv     results/sess-1b/ds000117/
cp    /data/teaching/costantinoai/sess-1b/ds000117/README    results/sess-1b/ds000117/
cp    /data/teaching/costantinoai/sess-1b/ds000117/CHANGES   results/sess-1b/ds000117/
chmod -R u+w results/sess-1b

### 1.2 🐍 A first look at the dataset

Let's see what we just copied.


In [ ]:
%%bash
# `tree` prints a folder and its contents as an indented hierarchy,
# so we can see the BIDS structure at a glance.
tree results/sess-1b/ds000117

Notice that each subject has a `ses-mri/` folder, matching the BIDS `sub-XX/ses-YY/...` pattern. 

Inside `ses-mri/func/` you will find **two BOLD runs** of the `task-facerecognition` task: `run-01` and `run-08`, together with their `events.tsv` files. The original study recorded 9 runs per subject; we kept two that we will compare later: a typical early run (`run-01`) and the run with the most head motion (`run-08`, near the end of the session when fatigue had set in).

The `derivatives/` subfolder is not here yet. It will appear after we run the next two copy cells, which populate `derivatives/mriqc/` (Section 2) and `derivatives/fmriprep/` (Section 3), simulating the effect of actually running those two BIDS Apps.

This is a common BIDS pattern: a single dataset can mix multiple imaging modalities, sessions, tasks, runs, and (under `derivatives/`) the outputs of many processing pipelines, all organised with the same filename convention you learned in sess-1a.

### 1.3 🐍 The experimental task, briefly

During each BOLD run, participants saw greyscale images of three kinds: **famous faces**, **unfamiliar faces**, and **scrambled faces** (phase-scrambled versions of the face images). We won't dig into the experimental design here: that belongs in the next session, when we fit a first-level GLM on this data. For now, just a quick look at what the subjects actually saw:

In [ ]:
from pathlib import Path             # object-oriented file paths
import pandas as pd                  # tabular data (TSV tables, confounds, IQMs)
import matplotlib.pyplot as plt      # static plots
from PIL import Image                # opening .bmp stimulus images

# Path to the raw BIDS dataset we just copied into results/
RAW_DIR = Path('results/sess-1b/ds000117')

# The dataset ships its task stimuli inside a BIDS-standard `stimuli/` folder.
# For the fMRI sessions the images live under stimuli/func/ and are named:
#   f###.bmp   famous face
#   u###.bmp   unfamiliar face
#   s###.bmp   scrambled face
stim_dir = RAW_DIR / 'stimuli' / 'func'

# Pick one example of each category, just to show what a participant saw.
# Keys are the category labels, values are the stimulus filenames.
examples = {
    'Famous face'     : 'f001.bmp',
    'Unfamiliar face' : 'u001.bmp',
    'Scrambled face'  : 's001.bmp',
}

# Create a small figure with 1 row and 3 columns (one panel per category)
fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))

# Loop over the three panels and fill each with one stimulus image
for ax, (label, fname) in zip(axes, examples.items()):
    ax.imshow(Image.open(stim_dir / fname), cmap='gray')   # BMPs are greyscale, so use a grey colormap
    ax.set_title(label)                                    # put the category name above the panel
    ax.axis('off')                                         # hide tick marks and axis labels (not informative for images)

# Adjust spacing between subplots so titles and borders do not overlap
plt.tight_layout()

# Save the figure to disk inside results/ for later reference
plt.savefig('results/sess-1b/stimuli_examples.png', dpi=120, bbox_inches='tight')

# Render the figure inline in the notebook
plt.show()

> 📖 For full details of the experimental design, see the original paper:
> Wakeman, D. G., & Henson, R. N. (2015). *A multi-subject, multi-modal human neuroimaging dataset.* Scientific Data, 2, 150001. [nature.com/articles/sdata20151](https://www.nature.com/articles/sdata20151)

### 1.4 📝 Practice: explore the BIDS folder

Before we dive into QC and preprocessing, take a moment to look at the raw dataset with your own eyes. In VS Code's **Explorer** panel on the left, expand:

```
results/
    └── sess-1b/
        └── ds000117/
```

Browse through the folder. You should recognise everything from sess-1a: top-level files like `participants.tsv`, `dataset_description.json`, `task-facerecognition_bold.json`, and one folder per subject.

> ❓ **Questions:**
> 1. Open `participants.tsv`. What columns does it contain, and how many subjects are there?
>
> 2. Open `task-facerecognition_bold.json`. What software was used for DICOM to nifti conversion? What is the TR?
>
> 3. Pick any one subject. Looking at `ses-mri/anat/` and `ses-mri/func/`, can you tell, just from the filenames, which file is the T1w and which are the BOLD runs?
>
> 4. Open the anatomical and one functional images using the Explorer tab on the left. Can you tell what is the matrix size of the two images? Which one has a higher spatial resolution?
>
> 5. Open one of the `sub-XX_ses-mri_task-facerecognition_run-XX_events.tsv` files. What do the columns describe, and what kind of experimental design does it encode?


## 2. Quality Control with MRIQC

### 2.1 Why QC before anything else?

Before running any preprocessing or statistical analysis, you should **look at your data**. Brain imaging is noisy, and problems that make results meaningless (excessive head motion, signal dropouts, coil failures, wrong coverage, ghosting) are often obvious in the raw images or in a handful of summary metrics. Detecting them early saves hours, or sometimes weeks, of analysis downstream.

For instance, have a look at these images we saw during our lecture on pre-processing:

<img src="assets/sess-1b/figures/lecture-mri-artifacts.png" width="700" style="border: 1.5px solid black;"/>

These images show common issues that we can spot early (and save ourselves quite some time) just by visually inspecting the images before preprocessing.

> ❓ **Question**: Can you identify what is the source of the artefacts in the images above?

However, some of the issues are not immediately visible upon visual inspection: they need to be quantified. This is where **[MRIQC](https://mriqc.readthedocs.io/)** comes in. MRIQC is a BIDS App that takes a BIDS dataset and produces, for each image, a set of **Image Quality Metrics (IQMs)** and an **HTML report** summarising them visually. It does not *modify* the data: it only *characterises* it.

### 2.2 What does MRIQC measure?

MRIQC computes dozens of metrics per image. Some examples:

| Metric | Modality | What it captures | Intuition |
|---|---|---|---|
| **SNR** (signal-to-noise ratio) | T1w, BOLD | signal inside tissue ÷ noise outside the brain | higher = better |
| **tSNR** (temporal SNR) | BOLD | mean ÷ standard deviation across time, per voxel | higher = cleaner time series |
| **FD** (framewise displacement) | BOLD | per-volume head movement in mm | lower = less motion; > 0.5 mm per volume is usually considered a problem |
| **DVARS** | BOLD | RMS change in image intensity between consecutive volumes | spikes indicate sudden signal changes, often motion-related |

You will meet **FD** and **DVARS** again later on: these are key "motion metrics" that connect MRIQC (QC of the *raw* data) with fMRIprep (QC of the *preprocessed* data).

### 2.3 The MRIQC command

MRIQC follows the standard BIDS App interface: `mriqc BIDS_DIR OUTPUT_DIR ANALYSIS_LEVEL [options]`. Notice the output folder: by BIDS convention, MRIQC's outputs go under the input dataset's `derivatives/mriqc/` subfolder. That is where we will point OUTPUT_DIR in the examples below.

A typical MRIQC call for a single subject looks like this:

```bash
# Participant level: one subject at a time, produces one HTML report per image
mriqc \
/path/to/ds000117 \                         # This is your BIDS input folder
/path/to/ds000117/derivatives/mriqc \       # Output folder (BIDS-derivatives convention)
participant \                               # Level. Must be participant, group
--participant-label 03 \                    # Participant(s) label(s)
--n-procs 4 --mem-gb 16                     # Here we set the available resources (4 cpu cores, 16 GB of RAM)
```

And once you have all the participants processed, you can run the group-level call:

```bash
# Group level: aggregates all participants into a group TSV and a group HTML report
mriqc \
/path/to/ds000117 \
/path/to/ds000117/derivatives/mriqc \
group                                       # Same as above, but different level
```

> **Note**: it is good practice to look at the documentation page of the tools you plan on using. For MRIQC, you can find the docs [here](https://mriqc.readthedocs.io/en/latest/usage.html#command-line-interface). 

Two things to notice. First, the **two analysis levels**: `participant` runs the per-image analysis; `group` pools the results into group-level summary TSVs and an HTML group report. You typically run participant-level in parallel on all subjects, then group-level once at the end. Second, `--participant-label` takes the subject ID **without the `sub-` prefix**: a BIDS App convention shared by all BIDS Apps.

On Neurodesk, MRIQC is available directly from the **Applications → Neurodesk → Functional Imaging → mriqc** menu, which opens a terminal with the MRIQC container loaded. You can also load it from any terminal with `ml mriqc/<version>`.

> ⏱️ **Runtime.** MRIQC typically takes ~20 minutes per subject (for our 9 runs). For 16 subjects that is several hours of compute. To save class time, **we will not run MRIQC now**: we copy the pre-computed outputs from the shared course folder and inspect them.

### 2.4 🐍 Get the MRIQC derivatives

The cell below copies the pre-computed MRIQC outputs into `results/sess-1b/ds000117/derivatives/mriqc/`, exactly where MRIQC would have written them if you had just finished running it.

In [ ]:
%%bash
# Same copy pattern as in Section 1, but for the MRIQC derivatives this time.
# We write them into the BIDS-standard location: <bids_root>/derivatives/mriqc/
mkdir -p results/sess-1b/ds000117/derivatives
rm -rf results/sess-1b/ds000117/derivatives/mriqc
cp -r /data/teaching/costantinoai/sess-1b/ds000117/derivatives/mriqc results/sess-1b/ds000117/derivatives/
chmod -R u+w results/sess-1b/ds000117/derivatives/mriqc

In [ ]:
%%bash
# Print the MRIQC derivatives folder as a tree
tree results/sess-1b/ds000117/derivatives/mriqc

Notice the structure:

- At the top level, one **`.html` report per image**: one per T1w, one per BOLD run. These are what you open to look at individual scans.
- Two **group-level tables**: `group_T1w.tsv` and `group_bold.tsv`. Each row is one image, each column is one IQM. These are great for plotting distributions and spotting outliers.
- Per-subject folders containing the raw IQM JSONs and figure data.

### 2.5 🐍 Exploring the group-level IQMs in Python

Let's read `group_bold.tsv` and look at how FD, tSNR, and DVARS are distributed across all subjects and runs.


In [ ]:
# Path to the MRIQC derivatives we just copied in
# (BIDS convention: <bids_root>/derivatives/mriqc/)
MRIQC_DIR = Path('results/sess-1b/ds000117/derivatives/mriqc')

# MRIQC writes one group-level TSV per modality. TSV = tab-separated table;
# `sep='\t'` tells pandas to split on tabs (not commas). One row per image,
# one column per IQM (image quality metric).
group_bold = pd.read_csv(MRIQC_DIR / 'group_bold.tsv', sep='\t')   # BOLD IQMs (one row per BOLD run)
group_T1w  = pd.read_csv(MRIQC_DIR / 'group_T1w.tsv',  sep='\t')   # T1w IQMs  (one row per T1w scan)

# Print how many scans each table contains
print("BOLD scans across whole dataset (runs * subjects):", len(group_bold))
print("T1w  scans (across all subjects):", len(group_T1w))
print()

# Preview a few key columns to build intuition. `bids_name` is the BIDS filename
# of the scan; the rest are IQMs (described in the table in the markdown above).
# `.head()` returns the first 5 rows of a DataFrame.
print("A few key columns for BOLD:")
print(group_bold[['bids_name', 'fd_mean', 'fd_perc', 'tsnr', 'dvars_nstd']].head())

In [ ]:
# Plot the distribution of three key IQMs across all BOLD runs in the dataset.
# A long right tail in fd_mean means some runs have noticeably more motion than the rest.

# The three IQMs we will plot, with a distinct colour and human-readable title each.
metrics = ['fd_mean', 'tsnr', 'dvars_nstd']
colors  = ['#d62728', '#1f77b4', '#2ca02c']                    # red, blue, green
titles  = ['Framewise Displacement (mm)',                       # fd_mean
           'Temporal SNR',                                      # tsnr
           'DVARS (normalised)']                                # dvars_nstd

# Create one figure with a row of len(metrics) panels (one per metric).
# `constrained_layout=True` handles spacing automatically, replacing `plt.tight_layout()`.
fig, axes = plt.subplots(1, len(metrics), figsize=(13, 3.8), constrained_layout=True)

# Fill each panel with the histogram of the corresponding metric
for ax, metric, color, title in zip(axes, metrics, colors, titles):
    values = group_bold[metric].dropna()                       # drop missing values before plotting

    # Histogram: `bins=25` splits the range into 25 buckets; `edgecolor='white'` draws bar borders.
    ax.hist(values, bins=25, color=color, edgecolor='white', alpha=0.85)

    # Vertical dashed line at the median, with the numeric value shown in the legend.
    median = values.median()
    ax.axvline(median, color='black', linestyle='--', linewidth=1, label=f'median = {median:.2f}')

    ax.set_title(f"{title}   (n = {len(values)})", fontsize=11)  # n = number of non-missing values
    ax.set_xlabel(metric)                                        # raw column name, for reference
    ax.set_ylabel('# runs')                                      # the y-axis is always a count of runs
    ax.legend(fontsize=9, loc='upper right')

    # Subtle horizontal grid, and hide the top/right spines for a cleaner look.
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Save the figure to disk for later reference
plt.savefig('results/sess-1b/iqm_distributions.png', dpi=120, bbox_inches='tight')

# Render the figure inline in the notebook
plt.show()

The distribution of `fd_mean` has a clear right tail: most runs sit around 0.1 mm, but a handful are well above 0.2 mm. We can use this to pick a **low-motion subject** ("good") and a **high-motion subject** ("bad") to compare in detail throughout the rest of the notebook.

> **Note on `n = 144`.** The MRIQC `group_bold.tsv` we just loaded summarises the **entire ds000117 cohort**: 16 subjects × 9 BOLD runs each = 144 scans. These group-level IQMs were pre-computed by OpenNeuroDerivatives on the full dataset, and that is what the histograms above are plotting. To keep this session light, our local copy of the **raw data and fMRIprep derivatives** only contains **two subjects** (sub-03 and sub-05) with **two runs each** (run-01 and run-08). But the group-level QC metrics still describe the whole cohort, which is what lets us pick representative "good" and "bad" subjects in the next cell.


In [ ]:
# Extract the 2-digit subject id from each BIDS filename using a regular expression.
# The pattern r'sub-([0-9]+)' captures the digits that appear right after 'sub-'.
df = group_bold.copy()                                        # work on a copy so we do not modify the original DataFrame
df['sub'] = df['bids_name'].str.extract(r'sub-([0-9]+)')      # add a new column with just the subject id

# Compute the mean fd_mean per subject (each subject has 9 runs), then sort
# the result from lowest to highest.
# `groupby('sub')` groups rows by subject id, `['fd_mean'].mean()` averages fd_mean
# within each group, and `.sort_values()` sorts the resulting Series ascending.
by_sub = df.groupby('sub')['fd_mean'].mean().sort_values()

# Print the full ranking so we can see the spread across the whole dataset
print("Subjects ranked by mean FD across all 9 runs:")
print(by_sub.to_string())

# Pick the extremes of the distribution: least and most motion.
GOOD_SUB = by_sub.index[0]     # first row after sort: lowest mean FD
BAD_SUB  = by_sub.index[-1]    # last  row after sort: highest mean FD

# Print the chosen subjects and their mean FD (in mm, to 3 decimals)
print()
print(f"GOOD = sub-{GOOD_SUB}  (mean FD = {by_sub.iloc[0]:.3f} mm)")
print(f"BAD  = sub-{BAD_SUB}  (mean FD = {by_sub.iloc[-1]:.3f} mm)")

These two subjects are our reference pair for the rest of the notebook.

### 2.6 🖥️ Reading an MRIQC HTML report

MRIQC's most important output is the **per-image HTML report**. Each bundles the IQMs with embedded visualisations (background mosaics, carpet plots, tiled slice views) that let you *see* what the numbers describe.

**Instructions:**

> **Note**: the first time you open an HTML file in Neurodesk, you need to set firefox browser as the default app to open `.html` files. Right click on the `.html` file --> Open with.. --> select Firefox --> check the option to use firefox as default app for HTML files

<img src="assets/sess-1b/figures/set-firefox-html.png" width="700" style="border: 1.5px solid black;"/>

1. Switch to **Neurodesktop**.
2. Open the file manager and navigate to `~/kul-hbi-workshop/results/sess-1b/ds000117/derivatives/mriqc/`.
3. Double-click `sub-05_ses-mri_task-facerecognition_run-01_bold.html` to open the **good** subject's report in the browser.
4. Then open `sub-03_ses-mri_task-facerecognition_run-01_bold.html` and compare.

Focus on these panels (more detail in the [MRIQC report docs](https://mriqc.readthedocs.io/en/latest/reports.html)):

- **Background mosaic**: a brightness-boosted view of the air around the head. In a clean scan it looks like uniform noise. Structured patterns here mean ghosting, spikes, or coil problems.
- **Zoomed-in mosaic**: a tiled view of all slices of the brain. Look for dropouts or wraps.
- **Carpet plot**: a 2D representation where each row is one voxel's time series. Vertical stripes = abrupt changes across the whole brain, a hallmark of motion.
- **IQMs table**: the numerical summary at the bottom.

### 2.7 📝 Practice: spot the difference

Open both HTML reports side by side (or in two browser tabs) and answer the questions below.

> ❓ **Questions:**
> 1. Which subject has the higher mean FD? By roughly what factor?
>
> 2. Compare the two **carpet plots**. What is structurally different between them?
>
> 3. Looking at the **background mosaic**, do you see any structured patterns in either subject? What would such patterns indicate?
>
> 4. Open the `group_bold.html` report (the group-level one at the top of the folder). Can you spot our two subjects in the group-level scatter plots? Where do they sit compared to the rest of the cohort?

## 3. Preprocessing with fMRIprep

Now that we know the quality of our raw data, we can preprocess it. Preprocessing turns the raw BOLD time series (a 4D volume of intensities, misaligned across time and distorted by several acquisition artefacts) into a **clean, analysis-ready time series** in a standard coordinate space.

### 3.1 What does fMRI preprocessing do?

A typical fMRI preprocessing pipeline performs, in sequence:

1. **Slice-timing correction**: the slices of a single volume are not acquired simultaneously but interleaved over the TR. Slice-timing correction temporally aligns them.
2. **Motion correction (realignment)**: the subject's head moves during scanning. Each volume is rigidly registered (6 parameters: 3 translations + 3 rotations) to a reference volume so the brain is in the same position across time. This is where the **6 Head Motion Parameters (6 HMP)** come from.
3. **Susceptibility distortion correction (SDC)**: BOLD images (echo-planar imaging) are geometrically distorted near air-tissue interfaces (frontal lobes, temporal poles). Fieldmaps, or synthetic fieldmap estimation, are used to undistort them.
4. **Coregistration to T1w**: the BOLD is rigidly aligned to the subject's own T1w anatomical.
5. **Normalisation to a template**: the T1w (and, via the BOLD-to-T1w mapping, the BOLD) is nonlinearly warped to a standard template (typically `MNI152NLin2009cAsym`) so analyses can be pooled across subjects.
6. **Resampling**: after computing all the above transforms, the BOLD is *resampled* to the target grid. For efficiency fMRIprep composes all the transforms and applies a **single resampling**, which limits blurring from repeated interpolation.
7. **Confound estimation**: fMRIprep does not clean the signal itself (that is your decision at the analysis stage), but it computes a large set of **nuisance regressors** (motion, aCompCor, CSF/WM signals, FD, DVARS, ...) and writes them to a `*_desc-confounds_timeseries.tsv` file for you to use downstream.

### 3.2 Why fMRIprep?

Before fMRIprep, each lab typically assembled its own preprocessing pipeline from FSL/SPM/ANTs/AFNI tools, configured differently across labs and projects: a major source of variability across papers. **[fMRIprep](https://fmriprep.org/)** was designed as an **opinionated, minimal preprocessing pipeline** that exposes only a handful of scientific decisions to the user and handles the hundreds of implementation details uniformly. It is BIDS-native (input *and* output), is packaged as a container (will run on any machine, and produce the same results, with no installation), is publicly maintained, and has been validated by thousands of studies.

### 3.3 The fMRIprep command

Here is a typical fMRIprep command. Like MRIQC, fMRIprep follows the BIDS App interface and writes its outputs under the input dataset's `derivatives/fmriprep/` subfolder by convention:

```bash
fmriprep \
/path/to/ds000117 \
/path/to/ds000117/derivatives/fmriprep \
participant \
--participant-label 03 \
--output-spaces MNI152NLin2009cAsym:res-2 T1w \
--fs-license-file /path/to/freesurfer.lic \
--fs-no-reconall \
--n-procs 8 --mem-mb 32000 \
-w /scratch/fmriprep-work
```

Key options:

- `--output-spaces`: the target spaces into which the BOLD is resampled. `MNI152NLin2009cAsym:res-2` means the standard 2 mm MNI template; `T1w` keeps a copy in the subject's native anatomical space.
- `--fs-license-file`: FreeSurfer is used for anatomical segmentation; it requires a (free) license file.
- `--fs-no-reconall`: skip the multi-hour FreeSurfer surface reconstruction. Fast, but no surface outputs. **We will use this flag in the home exercise** to keep runtime reasonable.
- `-w` / `--work-dir`: scratch directory for intermediate files (can be several GB per subject).

> ⏱️ **Runtime.** fMRIprep typically takes **several hours per subject** with `--fs-no-reconall`, and even longer with full FreeSurfer reconstruction. **We will not run fMRIprep now**: we copy the pre-computed outputs.

### 3.4 🐍 Get the fMRIprep derivatives

As before, we mirror the BIDS-derivatives layout: the fMRIprep outputs land in `results/sess-1b/ds000117/derivatives/fmriprep/`, next to `derivatives/mriqc/`.

Be patient, it will take a couple of minutes.

In [ ]:
%%bash
# Same copy pattern once more, now for the fMRIprep derivatives.
mkdir -p results/sess-1b/ds000117/derivatives
rm -rf results/sess-1b/ds000117/derivatives/fmriprep
cp -r /data/teaching/costantinoai/sess-1b/ds000117/derivatives/fmriprep results/sess-1b/ds000117/derivatives/
chmod -R u+w results/sess-1b/ds000117/derivatives/fmriprep

In [ ]:
%%bash
# Print the fMRIprep derivatives tree
tree results/sess-1b/ds000117/derivatives/fmriprep

The per-subject structure follows BIDS-Derivatives:

- `anat/`: preprocessed T1w, brain mask, tissue probability maps, transforms to MNI.
- `func/`: preprocessed BOLD in each output space, brain masks, and **the confounds TSV** (`*_desc-confounds_timeseries.tsv`), one per BOLD run.
- `figures/`: the per-subject SVG figures embedded in the HTML report.

> **Note on filenames.** The raw data uses two-digit run numbers (`run-01`, `run-02`, ...) but fMRIprep rewrites them with single digits (`run-1`, `run-2`, ...). This is a small BIDS inconsistency to keep in mind when writing globs.

To keep the shared folder small, we have pre-staged only **runs 1 and 2** of the preprocessed BOLD for each subject. The confounds and motion traces for those two runs are more than enough for our teaching purposes.

### 3.5 🐍 Loading the preprocessed BOLD

Let's load the preprocessed BOLD for the good subject in MNI space and do the same kind of inspection we did in sess-1a.


In [ ]:
from pathlib import Path
import nibabel as nib                 # loads NIfTI files (sess-1a)
from nilearn import plotting, image   # plotting + image-level operations on brain volumes
from bids import BIDSLayout           # query BIDS datasets by entity (sess-1a)

# Path to the fMRIprep derivatives we just copied in
# (BIDS convention: <bids_root>/derivatives/fmriprep/)
RAW_DIR = Path('results/sess-1b/ds000117')
FMRIPREP_DIR = Path('results/sess-1b/ds000117/derivatives/fmriprep')

# Build a BIDSLayout over both the raw dataset AND its fMRIprep derivatives.
# Passing `derivatives=FMRIPREP_DIR` tells pybids to also index the preprocessed
# outputs under their BIDS-Derivatives entities (space, res, desc, ...).
# `str(FMRIPREP_DIR)` converts the pathlib.Path to a plain string for pybids.
layout = BIDSLayout(RAW_DIR, derivatives=str(FMRIPREP_DIR))

# Print a summary of what the layout found (subjects, sessions, datatypes)
print(layout)

In [ ]:
# Helper function: ask the layout for the preprocessed BOLD of a given subject
# in MNI @ 2 mm space. `return_type='filename'` returns the file path as a
# plain string; `extension='.nii.gz'` rules out the accompanying JSON sidecar.
def preproc_bold(sub, run=1, space='MNI152NLin2009cAsym', res=2):
    """
    Retrieve the preprocessed BOLD fMRI file for a given subject/run.

    This function queries a BIDSLayout object (`layout`) for a preprocessed
    BOLD image in a specified standard space and resolution, and returns
    the first matching file path.

    Parameters
    ----------
    sub : str or int
        Subject identifier (e.g., '01').
    run : int, optional
        Run number to query (default is 1).
    space : str, optional
        Target space of the preprocessed image 
        (default: 'MNI152NLin2009cAsym').
    res : int, optional
        Spatial resolution in mm (default: 2).

    Returns
    -------
    str or None
        File path to the first matching preprocessed BOLD image (.nii.gz),
        or None if no match is found.
    """

    # Query BIDSLayout for preprocessed BOLD files matching the criteria
    hits = layout.get(
        subject=sub,
        run=run,
        space=space,
        res=res,
        desc='preproc',      # preprocessed data
        suffix='bold',       # BOLD fMRI images
        extension='.nii.gz', # exclude JSON sidecars
        return_type='filename'  # return file paths as strings
    )

    # Return the first match if available, otherwise None
    return hits[0] if hits else None

# Retrieve the preprocessed BOLD for our two subjects (run-1, MNI @ 2 mm)
good_bold = preproc_bold(GOOD_SUB)
bad_bold  = preproc_bold(BAD_SUB)

# Print the paths so we can see what was found
print('good:', good_bold)
print('bad :', bad_bold)

# Peek at the NIfTI header of the good subject, same fields we inspected in sess-1a
img = nib.load(good_bold)
print('shape   :', img.shape)               # (X, Y, Z, T); T = number of volumes
print('voxel mm:', img.header.get_zooms())  # (x, y, z, TR); the TR is the 4th value
print('TR (s)  :', img.header.get_zooms()[3])  # index 3 = TR (time between volumes, in seconds)

Now, since the BOLD image has 4 dimensions (remember: one 3-d volume across several TRs), we can compute the average of the volumes across time:

In [ ]:
# Average the 4D BOLD time series across time, giving a single 3D volume.
# For a preprocessed BOLD this should look like a grey-matter-highlighted EPI
# in the target MNI template (a good sanity check that we are in the right space).
mean_good = image.mean_img(good_bold, copy_header=True)    # mean across the time dimension

# Show the mean volume as three orthogonal slices
plotting.plot_epi(mean_good,
                  title=f'sub-{GOOD_SUB} - mean preprocessed BOLD (MNI)',
                  display_mode='ortho',           # axial, coronal, sagittal panels
                  cut_coords=(0, -10, 18))        # slice positions in MNI mm coordinates

# Render the figure inline in the notebook
plotting.show()

### 3.6 🐍 Head Motion Parameters (6 HMP)

The confounds TSV is where fMRIprep writes all its nuisance regressors. It is a tab-separated file with **one row per volume** and many columns, including the six motion parameters (`trans_x/y/z`, `rot_x/y/z`), `framewise_displacement`, `dvars`, aCompCor components, CSF and WM signals, and a `motion_outlier_*` column for each high-motion volume.

Let's load the confounds and plot the **6 HMP** for both subjects side by side.


In [ ]:
# Helper function: use pybids to locate the fMRIprep confounds TSV for a given
# subject/run, then read it with pandas.
# `desc='confounds'` + `suffix='timeseries'` + `extension='.tsv'` uniquely
# identifies the confounds file.
def confounds(sub, run=1):
    hits = layout.get(subject=sub, run=run,
                      desc='confounds', suffix='timeseries', extension='.tsv',
                      return_type='filename')
    return pd.read_csv(hits[0], sep='\t') if hits else None   # read the first match, or None if no match

# We deliberately use run-08 here: it is sub-03's noisiest run (mean FD ~0.36 mm,
# ~48% of volumes above MRIQC's 0.2 mm threshold) and gives the clearest
# good-vs-bad motion contrast below. run-01 of the same subjects is much calmer.
cg = confounds(GOOD_SUB, run=8)    # confounds table for the GOOD subject
cb = confounds(BAD_SUB,  run=8)    # confounds table for the BAD subject

# Quick sanity checks on what we just loaded
print('good confounds: shape =', cg.shape)           # (n_volumes, n_regressors)
print('first columns :', list(cg.columns)[:15])      # first 15 column names of the confounds table

In [ ]:
# The six head motion parameters written by fMRIprep.
# Translations (trans_*) are in mm, rotations (rot_*) in radians (both measured
# relative to the reference volume picked by fMRIprep during realignment).
HMP = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']

TRANS = ['trans_x', 'trans_y', 'trans_z']
ROT   = ['rot_x', 'rot_y', 'rot_z']

# Compute global symmetric y-limits separately for translations and rotations
tmax = max(abs(df[TRANS].values).max() for df in [cg, cb])
rmax = max(abs(df[ROT].values).max() for df in [cg, cb])

# Two stacked subplots: top = good subject, bottom = bad subject.
# `sharex=True` forces the two panels to use the same x-axis,
# while y-axes are manually synchronized across panels (mm left, rad right).
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)

# Loop over the two subjects, plotting all six HMPs in each panel.
# `zip(...)` pairs up the axes, dataframes, subject ids, and descriptive tags.
ax2_list = []  # keep track of secondary axes for legend

for ax, df, sub, tag in zip(axes, [cg, cb], [GOOD_SUB, BAD_SUB], ['low motion', 'high motion']):
    
    ax2 = ax.twinx()  # secondary y-axis for rotations (rad)
    ax2_list.append(ax2)

    for col in HMP:
        if col.startswith('trans'):
            ax.plot(df[col], label=col, linewidth=0.9)      # translations → left axis (mm)
        else:
            ax2.plot(df[col], linestyle='--', label=col, linewidth=0.9)  # rotations → right axis (rad)

    ax.set_title(f'sub-{sub} - 6 HMP  ({tag})')
    ax.set_ylabel('mm')                       # left axis: translations
    ax2.set_ylabel('rad')                     # right axis: rotations
    ax.axhline(0, color='k', linewidth=0.3)   # thin horizontal line at 0 for reference
    ax2.axhline(0, color='k', linewidth=0.3)

    # Enforce same symmetric limits across panels → aligns zero across both y-axes
    ax.set_ylim(-tmax, tmax)
    ax2.set_ylim(-rmax, rmax)

# Only the bottom panel needs an x-axis label (the top panel shares the x-axis)
axes[-1].set_xlabel('volume index')            # the x-axis is volume number, not seconds

# a single legend on the top panel, combining both axes
lines1, labels1 = axes[0].get_legend_handles_labels()
lines2, labels2 = ax2_list[0].get_legend_handles_labels()
axes[0].legend(lines1 + lines2, labels1 + labels2,
               ncol=6, fontsize=8, loc='upper right')

# Adjust spacing, save to disk, render inline
plt.tight_layout()
plt.savefig('results/sess-1b/hmp_good_vs_bad.png', dpi=120, bbox_inches='tight')
plt.show()

The bad subject's motion parameters show larger, more abrupt jumps than the good subject's. These six traces are exactly what fMRIprep used during the **realignment** step to rigidly re-register each volume to a reference; they are also the columns you typically include (or a subset, or their derivatives) as **nuisance regressors** in a first-level GLM to remove residual motion-related signal.

### 3.7 🐍 FD and DVARS

**Framewise Displacement (FD)** summarises the 6 motion parameters into a single per-volume scalar: how much (in mm) the head moved between this volume and the previous one, combining translations and rotations. **DVARS** measures how much the *whole-brain signal* changed from one volume to the next. Spikes in both usually co-occur.

A commonly used threshold for "problematic" volumes is **FD > 0.5 mm**.


In [ ]:
# 2x2 grid of panels: rows = metric (FD on top, DVARS on bottom), columns = subject (good, bad).
# `sharex='col'` makes the two rows in each column share the x-axis (volume index);
# `sharey='row'` makes the two columns in each row share the y-axis (same metric scale).
fig, axes = plt.subplots(2, 2, figsize=(12, 5.5), sharex='col', sharey='row')

# Loop over columns (one column per subject).
# `enumerate(...)` gives us col = 0 for the GOOD subject and col = 1 for the BAD subject.
for col, (df, sub, tag) in enumerate(zip([cg, cb], [GOOD_SUB, BAD_SUB], ['low motion', 'high motion'])):
    # Top row: FD time course, with the 0.5 mm problematic-volume threshold drawn in
    axes[0, col].plot(df['framewise_displacement'], color='C3', linewidth=0.9)
    axes[0, col].axhline(0.5, color='k', linestyle='--', linewidth=0.8, label='FD = 0.5 mm')   # threshold line
    axes[0, col].set_title(f'sub-{sub}  -  FD  ({tag})')
    axes[0, col].set_ylabel('FD (mm)')
    axes[0, col].legend(fontsize=8)

    # Bottom row: DVARS time course (no threshold line: DVARS thresholds vary by study)
    axes[1, col].plot(df['dvars'], color='C0', linewidth=0.9)
    axes[1, col].set_title(f'sub-{sub}  -  DVARS')
    axes[1, col].set_ylabel('DVARS')
    axes[1, col].set_xlabel('volume index')

# Adjust spacing, save to disk, render inline
plt.tight_layout()
plt.savefig('results/sess-1b/fd_dvars_good_vs_bad.png', dpi=120, bbox_inches='tight')
plt.show()

# Print how many volumes would be censored if we applied the 0.5 mm FD threshold.
# `(df['framewise_displacement'] > 0.5)` is a boolean Series; `.sum()` counts True values.
for df, sub in zip([cg, cb], [GOOD_SUB, BAD_SUB]):
    n   = int((df['framewise_displacement'] > 0.5).sum())   # count of volumes above threshold
    tot = len(df)                                           # total volumes in the run
    print(f"sub-{sub}: {n}/{tot} volumes with FD > 0.5 mm ({100*n/tot:.1f}%)")

If you were running a GLM on this data, you would typically **censor** (remove, or regress out with one-hot columns) the high-FD volumes using the `motion_outlier_*` columns fMRIprep already computed for you in the confounds TSV, or exclude runs with an average FD above the threshold,

### 3.8 🖥️ Reading the fMRIprep HTML report (post-preprocessing QC)

fMRIprep also produces a **per-subject HTML report** that is the post-preprocessing QC. Unlike MRIQC (which only describes the raw data), this report shows you **what fMRIprep actually did** to the data, and lets you verify each step visually. Almost every panel is an interactive SVG: hovering with the mouse fades between the *before* and *after* images.

Below are a few of the key panels, to give you a preview of what the report contains. The first four come from the **good** subject's report (sub-05, run-1); the last one (the carpet plot) is from the **bad** subject's noisy run (sub-03, run-8), where the effect of preprocessing is more visible.

**Anatomical tissue segmentation.** fMRIprep segments the T1w into grey matter, white matter, and CSF. The coloured contours should hug the correct anatomical boundaries:

<img src="assets/sess-1b/figures/sub-05_ses-mri_acq-mprage_dseg.svg" width="900" style="border: 1.5px solid black;"/>

**Spatial normalisation to MNI.** The T1w is nonlinearly warped to the `MNI152NLin2009cAsym` template. The red contour is the template boundary; it should closely follow the brain edge:

<img src="assets/sess-1b/figures/sub-05_ses-mri_acq-mprage_space-MNI152NLin2009cAsym_T1w.svg" width="900" style="border: 1.5px solid black;"/>

**BOLD-to-T1w coregistration (BBR).** The BOLD reference image is rigidly aligned to the subject's own T1w. Boundaries (grey-white) should line up between the two modalities:

<img src="assets/sess-1b/figures/sub-05_ses-mri_task-facerecognition_run-1_desc-bbregister_bold.svg" width="900" style="border: 1.5px solid black;"/>

**Susceptibility distortion correction (SDC).** Before/after EPI distortion correction. The *after* image should look anatomically closer to the T1w, especially in frontal and temporal regions:

<img src="assets/sess-1b/figures/sub-05_ses-mri_task-facerecognition_run-1_desc-sdc_bold.svg" width="900" style="border: 1.5px solid black;"/>

**Carpet plot of the preprocessed BOLD (sub-03, run-8, the noisy one).** The same visualisation we saw in MRIQC, but now on *preprocessed* data. Compare this with what the same run looks like *before* preprocessing in §3.9 below: the vertical stripes associated with head motion should be largely gone here:

<img src="assets/sess-1b/figures/sub-03_ses-mri_task-facerecognition_run-8_desc-carpetplot_bold.svg" width="900" style="border: 1.5px solid black;"/>

### 🖥️ Now open the real reports

The previews above are just static snapshots. The actual fMRIprep HTML report is interactive: hovering the mouse over each panel fades between the *before* and *after* images, and the per-run panels give you carpet plots, SDC previews, coregistration checks, and summary tables. Take a few minutes to open them:

1. Switch to **Neurodesktop**.
2. Navigate to `~/kul-hbi-workshop/results/sess-1b/ds000117/derivatives/fmriprep/` in the file manager.
3. Double-click **`sub-05.html`** (the good subject) to open it in the browser.
4. Then open **`sub-03.html`** (the bad subject) in another tab and compare.

Each report has been trimmed to the two runs we use in this session, **run-1** and **run-8**, plus the anatomical and field-mapping panels. As you scroll, focus on:

- **BBR coregistration** panels (run-1 and run-8 for each subject): do the grey/white boundaries still line up on both runs, or does the bad subject's run-8 look subtly off?
- **SDC** panels: does the *after* image look anatomically closer to the T1w in both runs?
- **Carpet plot** panels for run-1 vs run-8 of sub-03 (the bad subject): can you see the difference between a low-motion and a high-motion run on the *preprocessed* data? And how does preprocessing change the look of the bad-subject carpet compared to the raw carpet in §3.9?

Open both reports side by side (or in two browser tabs) and compare the good (`sub-05`) and bad (`sub-03`) subjects.

> ❓ **Questions:**
> 1. Compare the **BBR coregistration** panels for run-1 of each subject. Do the grey/white boundaries line up between BOLD and T1w just as well for the bad subject, or is there a visible misalignment?
>
> 2. Now compare **run-1 and run-8 for the bad subject** (sub-03): does the coregistration quality change between the calm early run and the noisy late run? What would you expect if motion during acquisition was the main driver?
>
> 3. Look at the **SDC** panels (before/after susceptibility distortion correction). Is the correction visibly helpful in both runs? Are there regions (e.g. frontal, temporal) where SDC makes a clearer difference?
>
> 4. Compare the **preprocessed carpet plots** for run-1 and run-8 of sub-03. Does preprocessing produce a similarly clean carpet in the noisy run, or do residual stripes remain? What does this tell you about how well preprocessing alone can "rescue" a high-motion run?
>
> 5. Based on what you see across these panels, would you include sub-03's runs in a group analysis? Would you include *some* of them but not others?


### 3.9 🐍 Before vs after: the carpet plot

The carpet plot above shows preprocessed data. For completeness, let's also compute it for the **raw** BOLD of our bad subject and compare the two side by side.

In [ ]:
from nilearn.plotting import plot_carpet          # carpet-plot routine from nilearn
from nilearn.masking import compute_epi_mask       # auto brain mask from a raw BOLD
import matplotlib.pyplot as plt                    # we also need plt directly for the extra FD panel

# Raw BOLD for the BAD subject. We pick run-08 (the noisy one) so the effect of
# realignment + resampling is visible in the preprocessed carpet below.
# `desc=None, space=None` tells pybids to return the *raw* file, not a derivative.
raw_bold = layout.get(subject=BAD_SUB, run=8, suffix='bold', extension='.nii.gz',
                      desc=None, space=None, return_type='filename')[0]

# Preprocessed BOLD for the same subject/run (MNI 2 mm, as loaded in §3.5)
pre_bold = preproc_bold(BAD_SUB, run=8)

# Brain mask produced by fMRIprep, in MNI space at 2 mm. This restricts the
# preprocessed carpet plot to in-brain voxels only, which makes the carpet
# much cleaner than showing the whole field of view.
mask_preproc = layout.get(subject=BAD_SUB, run=8, space='MNI152NLin2009cAsym', res=2,
                          desc='brain', suffix='mask', extension='.nii.gz',
                          return_type='filename')[0]

# The raw BOLD lives in scanner native space, so fMRIprep's MNI mask does not fit.
# `compute_epi_mask` estimates a brain mask directly from the raw 4D BOLD, giving
# us a native-space mask that makes the RAW carpet directly comparable with the
# preprocessed one (brain-only voxels on both sides).
mask_raw = compute_epi_mask(raw_bold)

# Framewise Displacement for this subject/run (we already loaded `cb` in §3.6).
fd = cb['framewise_displacement']

# Layout: 3 stacked panels with explicit height ratios.
#   Panel 1 (thin): FD time course for this run.
#   Panel 2 (tall): raw BOLD carpet.
#   Panel 3 (tall): preprocessed BOLD carpet.
# `sharex=True` keeps the volume axis aligned across the three panels.
fig, axes = plt.subplots(3, 1, figsize=(11, 7.5),
                         gridspec_kw={'height_ratios': [1, 3, 3]},
                         sharex=True)

# --- Top panel: FD time course ---
axes[0].plot(fd.values, color='C3', linewidth=0.9)                                      # red FD trace
axes[0].axhline(0.5, color='k', linestyle='--', linewidth=0.8, label='FD = 0.5 mm')    # threshold line
axes[0].set_ylabel('FD (mm)')
axes[0].set_title(f'sub-{BAD_SUB}, run-08  -  FD + raw vs preprocessed BOLD (PSC)', fontsize=11)
axes[0].legend(fontsize=8, loc='upper right')
axes[0].set_xlim(0, len(fd) - 1)                                                        # match the carpet volume range
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# --- Middle panel: raw BOLD carpet ---
# `standardize='psc'` expresses each voxel's timecourse as percent signal change
# from its own mean. Unlike z-score, PSC preserves the amplitude information:
# voxels whose motion spikes were large (in % terms) stay large on the colour
# scale, and preprocessing's reduction of those amplitudes becomes visible.
# `vmin`/`vmax` fix a common colour scale so the two carpets are directly
# comparable.
plot_carpet(raw_bold,
            mask_img=mask_raw,                      # brain-only voxels in scanner space
            axes=axes[1],
            title='RAW BOLD (PSC, brain-masked)',
            standardize='psc',
            vmin=-5, vmax=5)

# --- Bottom panel: preprocessed BOLD carpet (same PSC scale) ---
plot_carpet(pre_bold,
            mask_img=mask_preproc,                  # brain-only voxels in MNI space
            axes=axes[2],
            title='PREPROCESSED BOLD (PSC, brain-masked)',
            standardize='psc',
            vmin=-5, vmax=5)

# Save to disk and render inline
plt.tight_layout()
plt.savefig('results/sess-1b/carpet_before_after.png', dpi=120, bbox_inches='tight')
plt.show()

After preprocessing, some of the **vertical bands** in the raw carpet, each corresponding to a moment of head motion, are **attenuated**: realignment has brought every volume back into alignment, so the sudden whole-brain intensity changes are smaller.

But this subject had substantial motion, so **motion residuals are still clearly visible** in the preprocessed carpet. Notice how the remaining stripes line up with the spikes in the FD trace at the top of the figure: realignment corrects for motion **geometrically** (putting the brain back in the same position), but it cannot fully remove the **BOLD-signal disruption** that motion also causes (spin-history effects, local susceptibility changes, blood-flow fluctuations, and so on).

To mitigate these residual effects at the analysis stage, we typically include the **6 Head Motion Parameters** and **FD** as **nuisance regressors** in the first-level GLM, so their associated variance is removed from the BOLD time series before we estimate the effects of interest. We will see this in action in the next session's notebook, when we fit a GLM on ds000117.


### 3.10 🐍 A closer look at motion: peak-FD volume vs. two TRs earlier

Let's zoom in on a single moment of motion in the bad subject's noisy run. We pick the volume with the **highest FD** (`t`) and the volume **one TR earlier** (`t - 1`, just before the motion started), and look at their voxel-wise **signed** difference. We use the **preprocessed** BOLD (in MNI space) rather than the raw BOLD: because realignment has already been applied, the difference shows the **residual motion effect that preprocessing could not remove**. The difference map is then overlaid on the **subject's own T1 anatomical** (warped to the same MNI space by fMRIprep), which gives clear, subject-specific anatomical landmarks to interpret where those residuals concentrate.


In [ ]:
import nibabel as nib                                        # load 4D NIfTI and wrap 3D arrays
import numpy as np                                             # percentile + element-wise math
from nilearn import plotting                                   # interactive HTML viewer

# Load the PREPROCESSED 4D BOLD for our bad subject (sub-03, run-08).
# `pre_bold` was set in §3.9; it is in MNI 2 mm space.
pre_img  = nib.load(pre_bold)
pre_data = pre_img.get_fdata()                                  # 4D numpy array with shape (X, Y, Z, T)

# Use pybids to locate the SAME subject's preprocessed T1 in MNI 2 mm space.
# fMRIprep produces this file as part of the normalisation step, so it is
# perfectly aligned with the preprocessed BOLD loaded above.
t1_anat = layout.get(subject=BAD_SUB,
                     space='MNI152NLin2009cAsym', res=2,
                     desc='preproc', suffix='T1w', extension='.nii.gz',
                     return_type='filename')[0]
print('Subject T1 (MNI 2 mm):', t1_anat)

# Find the volume with the peak FD in this run. `cb` is the confounds table from §3.6.
# `.idxmax()` returns the integer index of the maximum value, skipping NaN entries.
t_peak   = int(cb['framewise_displacement'].idxmax())           # timepoint of maximum FD
t_before = max(t_peak - 1, 0)                                   # volume 1 TR earlier, clamped to 0
fd_peak  = cb['framewise_displacement'][t_peak]                 # FD value at the peak (mm)
print(f"Peak-FD volume : t    = {t_peak}  (FD = {fd_peak:.2f} mm)")
print(f"Reference      : t-2  = {t_before}")

# Extract the two 3D volumes and compute the SIGNED voxel-wise difference:
#   positive where voxel intensity increased from t-1 to t (red below)
#   negative where it decreased (blue below)
vol_before = pre_data[..., t_before]
vol_peak   = pre_data[..., t_peak]
vol_diff   = vol_peak - vol_before                              # SIGNED difference

# Wrap the numpy 3D array back into a NIfTI image for nilearn, reusing the
# affine/header of the 4D source so voxel-to-world coordinates stay correct.
img_diff = nib.Nifti1Image(vol_diff, pre_img.affine, pre_img.header)

# Threshold at the 98th percentile of |diff| so only the top ~2% of changing
# voxels light up in the overlay, and the rest stays transparent.
threshold = float(np.percentile(np.abs(vol_diff), 99))
print(f"Overlay threshold (98th percentile of |t - (t-1)|): {threshold:.1f}")

# Interactive viewer: signed difference overlaid on the SUBJECT's preprocessed
# T1 (in MNI space, so aligned with our BOLD). `vmax = threshold * 1.2` keeps
# the overlay saturated (otherwise diverging colour maps fade to near-white
# for values just above threshold).
vmax = threshold
plotting.view_img(
    img_diff,
    bg_img=t1_anat,                        # subject-specific anatomical background
    threshold=threshold,
    vmax=vmax,
    cmap='bwr',                            # pure blue / white / red, high saturation at the edges
    symmetric_cmap=True,
    colorbar=True,
    title=f'sub-{BAD_SUB}, run-08:  signed (t - (t-1)) on subject T1  (|thr| = {threshold:.0f})',
)

Look at the viewer above. The large positive/negative values do not sit uniformly across the brain: they concentrate in a thin band along the **edges of the brain**, around the **ventricles**, and at the **skull/brain interface**. That is the fingerprint of head motion. When the head shifts between two volumes, the fixed voxel grid now samples brain tissue where previously it sampled CSF (or vice versa), producing a large intensity change right at the tissue boundary. Voxels deep inside a homogeneous tissue type (middle of white matter, middle of a large grey-matter region) change much less, because they still sample the same tissue in both volumes.

This is exactly the signal disruption that realignment **cannot fully undo**. Once a volume has been acquired with motion, the **partial-volume mixing at the edges** is baked into the data. Realigning the volume brings the brain back into position, but the mis-sampled edge voxels stay mis-sampled, which is why we still see the pattern above even though the data in this viewer has already been preprocessed. That is also why the motion-related stripes in the preprocessed carpet above never fully disappear on high-motion runs, and why including motion regressors (6 HMP + FD) in the GLM, as we will do in the next notebook, is a **necessary complement** to preprocessing, not a substitute for it.

> ❓ **Questions:**
> 1. Some voxels in the overlay are **positive** (red: signal was higher at `t` than at `t-1`) while others are **negative** (blue: signal was lower at `t`). What is causing this sign difference? Think about what happens at a tissue boundary when the head moves by a fraction of a voxel.
>
> 2. Where are the highest-magnitude voxel signal changes **concentrated**, and where are they **absent**? Why would a uniformly distributed noise process *not* produce this pattern?


## 📝 Exercises

Now it is your turn. Using what you have learned about MRIQC and fMRIprep, complete the exercises below. These are **homework**: fMRIprep is too slow to run in class, so you will run it yourself on Neurodesk at home.

---

**Exercise 1: pick a new subject**

a. Pick **one subject** from `ds000117` other than `sub-03` and `sub-05` (the ones we used above).

b. Open the MRIQC group-level report at `results/sess-1b/ds000117/derivatives/mriqc/group_bold.html` and note where your subject sits in the distribution of `fd_mean` and `tsnr`. Is it a low-motion or a high-motion subject?

c. Open the MRIQC per-run HTML report for *one* of your subject's BOLD runs (in the same `derivatives/mriqc/` folder). Write 2-3 sentences describing what you see (IQMs, carpet plot, background).

---

**Exercise 2: run fMRIprep at home**

Now run fMRIprep yourself on the subject you just picked, on one or two BOLD runs.

a. Open **Neurodesktop**, open a terminal, and use DataLad to fetch the raw data for your subject:

```bash
cd ~/kul-hbi-workshop/results/sess-1b
datalad install -s ///openneuro/ds000117 ds000117-mine
datalad get -d ds000117-mine \
    ds000117-mine/sub-10/ses-mri/anat \
    ds000117-mine/sub-10/ses-mri/func/*run-01* \
    ds000117-mine/sub-10/ses-mri/func/*run-02*
```

Replace `10` with the subject you picked.

b. Load fMRIprep and run it with `--fs-no-reconall` to keep runtime manageable (~1-2 hours). Note the output path: it follows the BIDS-derivatives convention, writing into the input dataset's own `derivatives/fmriprep/` subfolder.

> - **FreeSurfer license**: fMRIprep requires the license file to be present. Request one for free at [surfer.nmr.mgh.harvard.edu/fswiki/License](https://surfer.nmr.mgh.harvard.edu/fswiki/License), and place the downloaded `.lic` license file in `/home/` before running the commands below in your terminal.

```bash
ml fmriprep/<version>

fmriprep \
    ~/kul-hbi-workshop/results/sess-1b/ds000117-mine \
    ~/kul-hbi-workshop/results/sess-1b/ds000117-mine/derivatives/fmriprep \
    participant \
    --participant-label 10 \
    --output-spaces MNI152NLin2009cAsym:res-2 \
    --fs-license-file ~/freesurfer.lic \
    --fs-no-reconall \
    --n-procs 4 --mem-mb 16000 \
    -w /tmp/fmriprep-work
```

c. When it finishes, open the HTML report at `~/kul-hbi-workshop/results/sess-1b/ds000117-mine/derivatives/fmriprep/sub-10.html` in the browser. Compare it with the reports of `sub-03` and `sub-05` you already know.

---

**Exercise 3: write a short QC judgement**

Using the fMRIprep HTML report (and, if helpful, the MRIQC report from Exercise 1), write a short QC judgement for your subject (3-5 sentences). Address at least:

a. How much did the subject move? What is the mean FD, and how many volumes are above the 0.5 mm threshold?

b. Does the coregistration between BOLD and T1w look reasonable?

c. Does the normalisation to MNI look reasonable?

d. Based on all of the above, would you include this subject in a group analysis? Why or why not?

---

**End of Session 1b.** In the next session we will move from preprocessed BOLD to first-level statistical analysis, using the same ds000117 dataset.